# LeetCode 950 — Reveal Cards In Increasing Order bằng Python

Link đề: https://leetcode.com/problems/reveal-cards-in-increasing-order/ · Độ khó: **Medium** · Chủ đề: Queue, Array, Sorting, Simulation

Skeleton LeetCode:

```python
class Solution:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        ...
```

Ghi chú về **đáp án duy nhất**: đề nói "return an ordering", nhưng vì các lá bài có giá trị **đôi một khác nhau** và thứ tự lật bắt buộc **tăng dần**, nên hoán vị trả về là **duy nhất** (ta sẽ chứng minh ở mục 10 và vét cạn ở mục 12D). Dù vậy notebook vẫn kiểm bằng **hàm kiểm tra tính hợp lệ** `reveals_increasing(order)` (mô phỏng quy trình lật bài) song song với so sánh trực tiếp. Điểm cốt lõi: **mô phỏng ngược** (đi từ lá lớn nhất, dựng lại bộ bài bằng deque) hoặc **mô phỏng xuôi trên chỉ số** rồi gán các giá trị đã sắp xếp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy trình lật bài: lật lá trên cùng, bỏ ra; nếu còn bài thì chuyển lá trên cùng kế tiếp xuống đáy; lặp lại.
2. Nắm công cụ Python: `collections.deque` (`appendleft`, `pop`, `popleft`, `append`), `sorted`, mô phỏng trên **chỉ số** thay vì trên giá trị.
3. Đi từ **thử mọi hoán vị** → **mô phỏng ngược bằng deque** → **mô phỏng xuôi trên chỉ số** → **công thức đệ quy giảm một nửa** → **mảng vòng tròn có công tắc bỏ qua**.
4. Hiểu mấu chốt: thứ tự các **vị trí** được lật là một hoán vị `π` chỉ phụ thuộc `n`, không phụ thuộc giá trị; vì vậy chỉ cần gán giá trị đã sắp xếp vào đúng các vị trí đó.
5. Tránh lỗi kinh điển: quên xoay lá đáy lên đầu trước khi đặt lá mới (mô phỏng ngược), xoay sai chiều, đặt lá **trước** khi xoay, duyệt giá trị tăng dần thay vì giảm dần, xoay khi bộ bài chỉ còn một lá, quên rằng `n` lẻ phải xoay phần còn lại.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `deck`. Mỗi lá bài có giá trị **duy nhất** `deck[i]`. Bạn được sắp xếp bộ bài theo **thứ tự bất kỳ**; ban đầu mọi lá đều úp.

Lặp lại các bước sau cho tới khi lật hết:

1. Lấy lá trên cùng, lật nó và bỏ ra khỏi bộ bài.
2. Nếu bộ bài còn lá nào, đặt lá trên cùng kế tiếp xuống **đáy** bộ bài.
3. Nếu còn lá chưa lật, quay lại bước 1; ngược lại dừng.

Trả về một thứ tự của bộ bài sao cho các lá được lật **theo thứ tự tăng dần**. Phần tử đầu của đáp án là lá trên cùng.

**Ví dụ 1:** `deck = [17,13,11,2,3,5,7]` → `[2,13,3,11,5,17,7]`.

**Ví dụ 2:** `deck = [1,1000]` → `[1,1000]`.

**Ràng buộc:** `1 <= deck.length <= 1000`; `1 <= deck[i] <= 10^6`; mọi `deck[i]` khác nhau.

In [1]:
import itertools
import math
import random
import time
from collections import deque
from typing import List

examples = [([17, 13, 11, 2, 3, 5, 7], [2, 13, 3, 11, 5, 17, 7]), ([1, 1000], [1, 1000]), ([5], [5]), ([3, 1, 2], [1, 3, 2])]
for d, want in examples:
    print(f"deck = {d} -> {want}")

deck = [17, 13, 11, 2, 3, 5, 7] -> [2, 13, 3, 11, 5, 17, 7]
deck = [1, 1000] -> [1, 1000]
deck = [5] -> [5]
deck = [3, 1, 2] -> [1, 3, 2]


## 3. Kiến thức nền

### 3.1 `collections.deque`

`deque` hỗ trợ thêm/bỏ ở **cả hai đầu** trong `O(1)`: `append`, `appendleft`, `pop`, `popleft`. Lật lá trên cùng = `popleft()`; chuyển lá trên cùng xuống đáy = `append(popleft())`; xoay ngược (lá đáy lên đầu) = `appendleft(pop())`.

In [2]:
dq = deque([1, 2, 3, 4])
print("popleft:", dq.popleft(), dq)
dq.append(dq.popleft())            # lá trên cùng xuống đáy
print("trên cùng -> đáy:", dq)
dq.appendleft(dq.pop())            # lá đáy lên đầu
print("đáy -> trên cùng:", dq)

popleft: 1 deque([2, 3, 4])
trên cùng -> đáy: deque([3, 4, 2])
đáy -> trên cùng: deque([2, 3, 4])


### 3.2 Hàm mô phỏng quy trình lật bài (kiểm tra tính hợp lệ)

`reveal_order(order)` thực hiện đúng 3 bước của đề và trả về dãy các lá theo thứ tự lật. Đây là **hàm kiểm tra tính hợp lệ**: một đáp án đúng ⇔ `reveal_order(order)` là dãy tăng dần (và là hoán vị của `deck`).

In [3]:
def reveal_order(order):
    """Mô phỏng nguyên văn đề: trả về các lá theo thứ tự được lật."""
    dq = deque(order)
    out = []
    while dq:
        out.append(dq.popleft())          # bước 1: lật lá trên cùng
        if dq:                            # bước 2: còn bài thì lá trên cùng kế tiếp xuống đáy
            dq.append(dq.popleft())
    return out


def reveals_increasing(order):
    out = reveal_order(order)
    return all(out[i] < out[i + 1] for i in range(len(out) - 1))


print(reveal_order([2, 13, 3, 11, 5, 17, 7]), reveals_increasing([2, 13, 3, 11, 5, 17, 7]), reveals_increasing([17, 13, 11, 2, 3, 5, 7]))

[2, 3, 5, 7, 11, 13, 17] True False


### 3.3 Mô phỏng trên chỉ số

Thứ tự lật các **vị trí** không phụ thuộc giá trị. Nếu mô phỏng quy trình trên dãy chỉ số `0, 1, …, n-1` ta được hoán vị `π`: lá thứ `k` được lật nằm ở vị trí `π[k]`. Sau đó chỉ cần `result[π[k]] = sorted(deck)[k]`.

In [4]:
def reveal_positions_sim(n):
    dq = deque(range(n))
    pos = []
    while dq:
        pos.append(dq.popleft())
        if dq:
            dq.append(dq.popleft())
    return pos


print("n = 7:", reveal_positions_sim(7))
print("n = 8:", reveal_positions_sim(8))

n = 7: [0, 2, 4, 6, 3, 1, 5]
n = 8: [0, 2, 4, 6, 1, 5, 3, 7]


## 4. Ý tưởng nền tảng / trực giác

`deck = [17,13,11,2,3,5,7]` đã sắp xếp: `[2,3,5,7,11,13,17]`. Với `n = 7` thứ tự lật **vị trí** là `0, 2, 4, 6, 3, 1, 5`. Lá nhỏ nhất `2` phải nằm ở vị trí `0`, lá nhỏ thứ hai `3` ở vị trí `2`, `5` ở vị trí `4`, `7` ở vị trí `6`, `11` ở vị trí `3`, `13` ở vị trí `1`, `17` ở vị trí `5`:

| vị trí | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| lá | 2 | 13 | 3 | 11 | 5 | 17 | 7 |

Đây chính là đáp án `[2,13,3,11,5,17,7]`. Hai cách nhìn:

- **Xuôi:** tính `π` bằng mô phỏng chỉ số, rồi điền các giá trị tăng dần.
- **Ngược:** đảo quy trình. Quy trình xuôi: *lật đầu, đáy ← lá kế*. Quy trình ngược khi đặt lá lớn → nhỏ: *đưa lá đáy lên đầu, rồi đặt lá mới lên đầu*.

In [5]:
d = [17, 13, 11, 2, 3, 5, 7]
s = sorted(d)
pi = reveal_positions_sim(len(d))
res = [None] * len(d)
for k, p in enumerate(pi):
    res[p] = s[k]
print("pi =", pi, "->", res)
assert res == [2, 13, 3, 11, 5, 17, 7]

pi = [0, 2, 4, 6, 3, 1, 5] -> [2, 13, 3, 11, 5, 17, 7]


## 5. Brute force — thử mọi hoán vị

**Ý tưởng:** thử mọi hoán vị của `deck`, lấy hoán vị đầu tiên mà `reveals_increasing` đúng. Chỉ dùng được cho `n ≤ 8` (`8! = 40320`); với `n = 1000` là bất khả thi. Nhưng đây là **oracle độc lập** cho mọi cách sau.

In [6]:
def reveal_brute(deck):
    found = None
    for perm in itertools.permutations(deck):
        if reveals_increasing(perm):
            if found is None:
                found = list(perm)
            else:
                raise AssertionError("có nhiều hơn một đáp án?!")      # để kiểm tra tính duy nhất
    return found


for d, want in examples:
    assert reveal_brute(d) == want, (d, reveal_brute(d))
print([reveal_brute(d) for d, _ in examples])

[[2, 13, 3, 11, 5, 17, 7], [1, 1000], [5], [1, 3, 2]]


In [7]:
# Code nộp LeetCode cho Brute force (chỉ để minh hoạ; sẽ TLE khi n lớn)
import itertools
from collections import deque


class SolutionBrute:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        for perm in itertools.permutations(deck):
            dq, out = deque(perm), []
            while dq:
                out.append(dq.popleft())
                if dq:
                    dq.append(dq.popleft())
            if all(out[i] < out[i + 1] for i in range(len(out) - 1)):
                return list(perm)
        return []

**Độ phức tạp:** `O(n! · n)` thời gian, `O(n)` bộ nhớ. Vô dụng với `n = 1000`.

**Nút thắt:** ta thử cả `n!` thứ tự trong khi thứ tự lật các **vị trí** là một hoán vị cố định chỉ phụ thuộc `n`. Chỉ cần tìm hoán vị đó.

## 6. Bước đột phá

Thứ tự lật cho `n` lá là một hoán vị `π(n)` của các vị trí; lá nhỏ thứ `k` đặt tại `π[k]`. Tính `π` bằng mô phỏng `O(n)` (xuôi), hoặc dựng bộ bài trực tiếp bằng quy trình **đảo ngược** (ngược), đều `O(n log n)` với bước sắp xếp.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** mô phỏng ngược với deque, **Cách 3** mô phỏng xuôi trên chỉ số, **Cách 4** công thức đệ quy giảm một nửa, **Cách 5** mảng vòng tròn có công tắc bỏ qua.

### Cách 2 — Mô phỏng ngược bằng deque (đáp án chính)

**Giải thích từng đoạn:**

1. Sắp xếp `deck` **giảm dần**; `dq = deque()` rỗng.
2. Với từng giá trị `x` từ lớn đến nhỏ: nếu `dq` không rỗng, đưa lá **đáy** lên đầu (`dq.appendleft(dq.pop())`) — đảo bước "chuyển lá kế xuống đáy"; sau đó đặt `x` lên **đầu** (`dq.appendleft(x)`) — đảo bước "lật lá trên cùng".
3. Kết quả là `list(dq)`.

In [8]:
def reveal_reverse(deck):
    dq = deque()
    for x in sorted(deck, reverse=True):          # lá lớn nhất được lật CUỐI CÙNG nên đặt trước
        if dq:
            dq.appendleft(dq.pop())               # đảo bước 2: lá đáy trở về đầu
        dq.appendleft(x)                          # đảo bước 1: lá vừa lật nằm trên cùng
    return list(dq)


for d, want in examples:
    assert reveal_reverse(d) == want
print([reveal_reverse(d) for d, _ in examples])

[[2, 13, 3, 11, 5, 17, 7], [1, 1000], [5], [1, 3, 2]]


In [9]:
# Code nộp LeetCode cho Cách 2
from collections import deque


class SolutionReverse:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        dq = deque()
        for x in sorted(deck, reverse=True):
            if dq:
                dq.appendleft(dq.pop())
            dq.appendleft(x)
        return list(dq)

**Độ phức tạp:** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ. Phần deque `O(n)`.

### Cách 3 — Mô phỏng xuôi trên chỉ số

**Ý tưởng:** `dq = deque(range(n))` chứa các **vị trí**; với mỗi giá trị đã sắp xếp tăng dần: `result[dq.popleft()] = x`; nếu còn thì `dq.append(dq.popleft())`.

In [10]:
def reveal_forward(deck):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):                        # lá nhỏ nhất được lật đầu tiên
        res[dq.popleft()] = x                     # vị trí được lật nhận giá trị x
        if dq:
            dq.append(dq.popleft())               # vị trí kế tiếp xuống đáy
    return res


for d, want in examples:
    assert reveal_forward(d) == want
print([reveal_forward(d) for d, _ in examples])

[[2, 13, 3, 11, 5, 17, 7], [1, 1000], [5], [1, 3, 2]]


In [11]:
# Code nộp LeetCode cho Cách 3
from collections import deque


class SolutionForward:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        n = len(deck)
        res = [0] * n
        dq = deque(range(n))
        for x in sorted(deck):
            res[dq.popleft()] = x
            if dq:
                dq.append(dq.popleft())
        return res

**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ.

### Cách 4 — Công thức đệ quy giảm một nửa

**Ý tưởng:** vòng đầu tiên lật các vị trí chẵn `0, 2, 4, …` (số lượng `⌈n/2⌉`) và chuyển các vị trí lẻ xuống đáy. Các vị trí lẻ còn lại `1, 3, 5, …` trở thành bộ bài mới kích thước `m = ⌊n/2⌋`:

- nếu `n` **chẵn**: lá cuối cùng được lật là `n-2`, rồi `n-1` bị chuyển xuống đáy ⇒ bộ bài mới giữ nguyên thứ tự `[1, 3, …, n-1]`;
- nếu `n` **lẻ**: lá cuối cùng được lật là `n-1`, rồi lá trên cùng kế tiếp (vị trí `1`) bị chuyển xuống đáy ⇒ bộ bài mới là `[3, 5, …, n-2, 1]` (xoay trái một bước).

Thứ tự lật = các vị trí chẵn rồi thứ tự lật của bộ bài mới (đệ quy), ánh xạ qua danh sách `rest`.

In [12]:
def reveal_positions_rec(n):
    if n == 0:
        return []
    if n == 1:
        return [0]
    evens = list(range(0, n, 2))                  # vòng 1 lật các vị trí chẵn
    rest = list(range(1, n, 2))                   # các vị trí lẻ bị chuyển xuống đáy
    if n % 2 == 1:                                # n lẻ: lá lẻ đầu tiên bị chuyển xuống đáy thêm một lần
        rest = rest[1:] + rest[:1]
    sub = reveal_positions_rec(len(rest))         # bài toán con kích thước n // 2
    return evens + [rest[j] for j in sub]


def reveal_halving(deck):
    res = [0] * len(deck)
    for x, p in zip(sorted(deck), reveal_positions_rec(len(deck))):
        res[p] = x
    return res


assert all(reveal_positions_rec(n) == reveal_positions_sim(n) for n in range(0, 200))
for d, want in examples:
    assert reveal_halving(d) == want
print("n = 7 :", reveal_positions_rec(7), "| n = 8:", reveal_positions_rec(8))

n = 7 : [0, 2, 4, 6, 3, 1, 5] | n = 8: [0, 2, 4, 6, 1, 5, 3, 7]


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionHalving:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        def positions(n):
            if n <= 1:
                return list(range(n))
            evens = list(range(0, n, 2))
            rest = list(range(1, n, 2))
            if n % 2:
                rest = rest[1:] + rest[:1]
            return evens + [rest[j] for j in positions(len(rest))]

        res = [0] * len(deck)
        for x, p in zip(sorted(deck), positions(len(deck))):
            res[p] = x
        return res

**Độ phức tạp:** `T(n) = T(n/2) + O(n)` ⇒ `O(n)` cho phần tính vị trí, tổng `O(n log n)` do sắp xếp; độ sâu đệ quy `O(log n)`.

### Cách 5 — Mảng vòng tròn có công tắc bỏ qua

**Ý tưởng:** duyệt vòng tròn trên mảng kết quả `res` (ô trống = `None`). Giữ công tắc `skip`: gặp ô trống, nếu `skip` sai thì **đặt** giá trị nhỏ nhất còn lại; sau đó đảo `skip`. Đây chính là quy trình "lật một, bỏ qua một" nhưng không cần deque: ô đã lấp không được tính.

In [14]:
def reveal_skip(deck):
    n = len(deck)
    s = sorted(deck)
    res = [None] * n
    i = 0
    skip = False
    placed = 0
    while placed < n:
        if res[i] is None:                        # chỉ ô trống mới "nằm trong bộ bài"
            if not skip:
                res[i] = s[placed]                # lá này sẽ được lật: nhận giá trị nhỏ nhất còn lại
                placed += 1
            skip = not skip                       # lật xong thì lá kế bị bỏ qua (xuống đáy) và ngược lại
        i = (i + 1) % n
    return res


for d, want in examples:
    assert reveal_skip(d) == want
print([reveal_skip(d) for d, _ in examples])

[[2, 13, 3, 11, 5, 17, 7], [1, 1000], [5], [1, 3, 2]]


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionSkip:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        n = len(deck)
        s = sorted(deck)
        res = [None] * n
        i, skip, placed = 0, False, 0
        while placed < n:
            if res[i] is None:
                if not skip:
                    res[i] = s[placed]
                    placed += 1
                skip = not skip
            i = (i + 1) % n
        return res

**Độ phức tạp:** mỗi vòng quét `O(n)`, số vòng `O(log n)` (mỗi vòng lấp khoảng một nửa số ô trống còn lại) nên `O(n log n)`; sắp xếp `O(n log n)`; bộ nhớ `O(n)`.

## 8. TRACE từng bước

**Trace Cách 2 (mô phỏng ngược)** với `deck = [17,13,11,2,3,5,7]`, sắp giảm dần `17, 13, 11, 7, 5, 3, 2`:

| lá `x` | đáy → đầu | sau khi `appendleft(x)` |
|---|---|---|
| 17 | (rỗng) | `[17]` |
| 13 | `[17]` | `[13, 17]` |
| 11 | `[17, 13]` | `[11, 17, 13]` |
| 7 | `[13, 11, 17]` | `[7, 13, 11, 17]` |
| 5 | `[17, 7, 13, 11]` | `[5, 17, 7, 13, 11]` |
| 3 | `[11, 5, 17, 7, 13]` | `[3, 11, 5, 17, 7, 13]` |
| 2 | `[13, 3, 11, 5, 17, 7]` | `[2, 13, 3, 11, 5, 17, 7]` |

Kết quả `[2,13,3,11,5,17,7]` ✓. Hàm trace dưới đây in từng bước và tự kiểm tra bằng `reveals_increasing`:

In [16]:
def trace_reverse(deck):
    print(f"deck = {deck}")
    dq = deque()
    for x in sorted(deck, reverse=True):
        before = list(dq)
        if dq:
            dq.appendleft(dq.pop())
        mid = list(dq)
        dq.appendleft(x)
        print(f"  x={x:>3}: bộ bài {before} -> đáy lên đầu {mid} -> đặt {x} lên đầu {list(dq)}")
    out = list(dq)
    print("=>", out, "| lật ra:", reveal_order(out))
    assert reveals_increasing(out)
    return out


assert trace_reverse([17, 13, 11, 2, 3, 5, 7]) == [2, 13, 3, 11, 5, 17, 7]

deck = [17, 13, 11, 2, 3, 5, 7]
  x= 17: bộ bài [] -> đáy lên đầu [] -> đặt 17 lên đầu [17]
  x= 13: bộ bài [17] -> đáy lên đầu [17] -> đặt 13 lên đầu [13, 17]
  x= 11: bộ bài [13, 17] -> đáy lên đầu [17, 13] -> đặt 11 lên đầu [11, 17, 13]
  x=  7: bộ bài [11, 17, 13] -> đáy lên đầu [13, 11, 17] -> đặt 7 lên đầu [7, 13, 11, 17]
  x=  5: bộ bài [7, 13, 11, 17] -> đáy lên đầu [17, 7, 13, 11] -> đặt 5 lên đầu [5, 17, 7, 13, 11]
  x=  3: bộ bài [5, 17, 7, 13, 11] -> đáy lên đầu [11, 5, 17, 7, 13] -> đặt 3 lên đầu [3, 11, 5, 17, 7, 13]
  x=  2: bộ bài [3, 11, 5, 17, 7, 13] -> đáy lên đầu [13, 3, 11, 5, 17, 7] -> đặt 2 lên đầu [2, 13, 3, 11, 5, 17, 7]
=> [2, 13, 3, 11, 5, 17, 7] | lật ra: [2, 3, 5, 7, 11, 13, 17]


**Trace Cách 3 (mô phỏng xuôi trên chỉ số)** với `n = 7`, giá trị tăng dần `2,3,5,7,11,13,17`:

| lá | hàng đợi chỉ số trước | lật vị trí | gán | hàng đợi sau khi đẩy xuống đáy |
|---|---|---|---|---|
| 2 | `[0,1,2,3,4,5,6]` | 0 | `res[0]=2` | `[2,3,4,5,6,1]` |
| 3 | `[2,3,4,5,6,1]` | 2 | `res[2]=3` | `[4,5,6,1,3]` |
| 5 | `[4,5,6,1,3]` | 4 | `res[4]=5` | `[6,1,3,5]` |
| 7 | `[6,1,3,5]` | 6 | `res[6]=7` | `[3,5,1]` |
| 11 | `[3,5,1]` | 3 | `res[3]=11` | `[1,5]` |
| 13 | `[1,5]` | 1 | `res[1]=13` | `[5]` |
| 17 | `[5]` | 5 | `res[5]=17` | (rỗng) |

In [17]:
def trace_forward(deck):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        before = list(dq)
        p = dq.popleft()
        res[p] = x
        if dq:
            dq.append(dq.popleft())
        print(f"  lá {x:>3}: hàng đợi {before} -> lật vị trí {p}, res[{p}] = {x}, hàng đợi sau {list(dq)}")
    print("=>", res)
    return res


assert trace_forward([17, 13, 11, 2, 3, 5, 7]) == [2, 13, 3, 11, 5, 17, 7]

  lá   2: hàng đợi [0, 1, 2, 3, 4, 5, 6] -> lật vị trí 0, res[0] = 2, hàng đợi sau [2, 3, 4, 5, 6, 1]
  lá   3: hàng đợi [2, 3, 4, 5, 6, 1] -> lật vị trí 2, res[2] = 3, hàng đợi sau [4, 5, 6, 1, 3]
  lá   5: hàng đợi [4, 5, 6, 1, 3] -> lật vị trí 4, res[4] = 5, hàng đợi sau [6, 1, 3, 5]
  lá   7: hàng đợi [6, 1, 3, 5] -> lật vị trí 6, res[6] = 7, hàng đợi sau [3, 5, 1]
  lá  11: hàng đợi [3, 5, 1] -> lật vị trí 3, res[3] = 11, hàng đợi sau [1, 5]
  lá  13: hàng đợi [1, 5] -> lật vị trí 1, res[1] = 13, hàng đợi sau [5]
  lá  17: hàng đợi [5] -> lật vị trí 5, res[5] = 17, hàng đợi sau []
=> [2, 13, 3, 11, 5, 17, 7]


## 9. Code nộp LeetCode

Chọn **Cách 2** (mô phỏng ngược): ngắn, không cần chỉ số. Giữ nguyên `class Solution`, `deckRevealedIncreasing(self, deck)`:

In [18]:
class Solution:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        dq = deque()
        for x in sorted(deck, reverse=True):       # đi từ lá lớn nhất (lật cuối) đến nhỏ nhất (lật đầu)
            if dq:
                dq.appendleft(dq.pop())            # đảo bước "lá kế xuống đáy": lá đáy trở về đầu
            dq.appendleft(x)                       # đảo bước "lật lá trên cùng"
        return list(dq)


print([Solution().deckRevealedIncreasing(list(d)) for d, _ in examples])
assert all(Solution().deckRevealedIncreasing(list(d)) == w for d, w in examples)

[[2, 13, 3, 11, 5, 17, 7], [1, 1000], [5], [1, 3, 2]]


## 10. Vì sao đúng

**(a) Đáp án duy nhất.** Quy trình lật không nhìn giá trị: với `n` lá nó lật các **vị trí** theo một thứ tự cố định `π(n)`. Lá được lật thứ `k` phải là lá nhỏ thứ `k` (đòi hỏi thứ tự tăng; giá trị đôi một khác nhau nên "tăng dần" ⇔ "đúng thứ tự sắp xếp"). Vậy `order[π[k]] = sorted(deck)[k]`: xác định duy nhất hoán vị trả về.

**(b) Cách 3 đúng.** Mô phỏng quy trình trên dãy chỉ số `0..n-1` cho đúng `π(n)`; gán các giá trị tăng dần theo `π` nghĩa là lá thứ `k` được lật chính là `sorted[k]`.

**(c) Cách 2 đúng (mô phỏng ngược).** Gọi `D_k` là bộ bài (đã sắp đúng) chỉ gồm `k` lá lớn nhất. Ta chứng minh quy nạp rằng sau khi xử lý `k` lá lớn nhất, `dq = D_k` — bộ bài sắp đúng cho `k` lá đó.
- `k = 1`: `[x]` ✓.
- Bước `k → k+1`: bộ bài đúng cho `k+1` lá, `D_{k+1} = [y, …]` với `y` là lá nhỏ nhất. Sau khi lật `y` và chuyển lá kế xuống đáy ta được `D_k` (các lá còn lại). Do đó `D_{k+1}` = (lá `y`) ghép với bộ bài `E` mà sau thao tác "chuyển đầu xuống đáy" bằng `D_k`. Phép ngược của "đầu xuống đáy" là "đáy lên đầu" ⇒ `E = rotate_right(D_k)`; `D_{k+1} = [y] + E`. Đó đúng là hai dòng `appendleft(pop())` rồi `appendleft(y)`.
- Với `D_k` rỗng (hoặc một lá) không cần xoay: xoay một deque một lá không đổi gì, và dòng `if dq` bảo vệ trường hợp rỗng.

**(d) Cách 4 đúng.** Vòng 1 lật các vị trí chẵn vì cặp (lật, chuyển) tiêu thụ hai vị trí liên tiếp. Phần còn lại sau vòng 1 là các vị trí lẻ theo thứ tự `1, 3, 5, …`; nếu `n` lẻ thì lá cuối cùng được lật là `n-1` và sau đó vị trí `1` (lá trên cùng kế tiếp) bị chuyển xuống đáy ⇒ xoay trái một bước; nếu `n` chẵn thì `n-1` đã bị chuyển xuống đáy ở bước cuối nên không xoay. Bộ bài mới có `n // 2` lá và quy trình tiếp diễn **đồng dạng**, nên đệ quy ánh xạ qua `rest`. Đã kiểm `reveal_positions_rec(n) == reveal_positions_sim(n)` cho `n < 200` ở mục 7 và cho `n ≤ 1000` ở mục 12.

**(e) Cách 5 đúng.** `res` có ô trống đại diện các lá chưa lật còn trong bộ bài; duyệt vòng tròn giữ đúng thứ tự "đầu → đáy" của bộ bài mô phỏng. Công tắc `skip` luân phiên: ô đầu tiên được lật (đặt giá trị), ô kế tiếp bị chuyển xuống đáy (bỏ qua, nhưng vẫn là ô trống để gặp lại ở vòng sau). Ô đã lấp bị bỏ qua không tính (đã rời bộ bài).

**(f) Kiểm chứng bằng máy.** Với mọi `n ≤ 8`, mọi hoán vị của `1..n` (hơn `46 000` hoán vị) có **đúng một** hoán vị lật tăng dần (mục 12D); mọi cách khớp hoán vị đó và khớp lẫn nhau với `n` tới `1000`.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi hoán vị | `O(n! · n)` | `O(n)` | oracle, `n ≤ 8` |
| 2. Mô phỏng ngược (deque) | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. Mô phỏng xuôi chỉ số | `O(n log n)` | `O(n)` | rõ ý nghĩa `π` |
| 4. Đệ quy giảm một nửa | `O(n log n)` | `O(n)` | phần vị trí `O(n)` |
| 5. Mảng vòng + công tắc | `O(n log n)` | `O(n)` | không cần deque |

Đo thực tế với `n = 1000` (trung bình 20 lần):

In [19]:
def bench(fn, deck, repeat=20):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(deck)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(950)
big = rng.sample(range(1, 10 ** 6 + 1), 1000)
for name, fn in [("reverse", reveal_reverse), ("forward", reveal_forward), ("halving", reveal_halving), ("skip", reveal_skip)]:
    print(f"{name:8s}: {bench(fn, big) * 1e3:7.3f} ms (n = 1000)")

reverse :   0.146 ms (n = 1000)
forward :   0.146 ms (n = 1000)
halving :   0.178 ms (n = 1000)


skip    :   0.790 ms (n = 1000)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn **mọi hoán vị** `n ≤ 8` (kiểm tính duy nhất), (E) tính chất / ngẫu nhiên với seed cố định kèm hàm kiểm tra hợp lệ, (F) giới hạn `n = 1000`.

### A. Ví dụ của đề

Vì đáp án duy nhất ta so sánh trực tiếp, **và** kiểm bằng hàm hợp lệ `reveals_increasing` + `sorted(order) == sorted(deck)`.

In [20]:
def valid_answer(deck, order):
    """Hàm kiểm tra tính hợp lệ: hoán vị của deck và lật ra theo thứ tự tăng dần."""
    return sorted(order) == sorted(deck) and reveals_increasing(order)


IMPLS = {
    "reverse": reveal_reverse,
    "forward": reveal_forward,
    "halving": reveal_halving,
    "skip": reveal_skip,
    "Solution": lambda d: Solution().deckRevealedIncreasing(list(d)),
    "SolutionReverse": lambda d: SolutionReverse().deckRevealedIncreasing(list(d)),
    "SolutionForward": lambda d: SolutionForward().deckRevealedIncreasing(list(d)),
    "SolutionHalving": lambda d: SolutionHalving().deckRevealedIncreasing(list(d)),
    "SolutionSkip": lambda d: SolutionSkip().deckRevealedIncreasing(list(d)),
}


def check(deck, want=None, names=None, brute=True):
    for name in (names or IMPLS):
        got = IMPLS[name](list(deck))
        assert valid_answer(deck, got), (name, deck, got)
        if want is not None:
            assert got == want, (name, deck, got, want)
    if brute and len(deck) <= 7:
        assert reveal_brute(deck) == IMPLS["reverse"](list(deck))


for d, want in examples:
    check(d, want)
print("ví dụ đề: brute force và", len(IMPLS), "cài đặt đúng, kết quả hợp lệ")

ví dụ đề: brute force và 9 cài đặt đúng, kết quả hợp lệ


### B. Edge case

- `n = 1`: `[5]` → `[5]`; `n = 2`: `[1000, 1]` → `[1, 1000]`.
- `n = 3`: `[3,1,2]` → `[1,3,2]`.
- Đã sắp tăng, đã sắp giảm; giá trị lớn nhất `10^6`; `n` là luỹ thừa của 2 (`4`, `8`, `16`) và `n` lẻ.
- Với `n = 4`: `[1,2,3,4]` → `[1,3,2,4]`.
- Với `n = 5`: `[1,2,3,4,5]` → `[1,5,2,4,3]`.

In [21]:
cases = [
    ([5], [5]), ([1000, 1], [1, 1000]), ([3, 1, 2], [1, 3, 2]), ([1, 2, 3, 4], [1, 3, 2, 4]), ([4, 3, 2, 1], [1, 3, 2, 4]),
    ([1, 2, 3, 4, 5], [1, 5, 2, 4, 3]), ([10 ** 6], [10 ** 6]), ([10 ** 6, 1], [1, 10 ** 6]),
    ([8, 7, 6, 5, 4, 3, 2, 1], None), ([1, 2, 3, 4, 5, 6, 7], None), (list(range(16, 0, -1)), None),
]
for d, want in cases:
    check(d, want)
print(len(cases), "edge case: mọi cài đặt đúng")
print("n = 8 :", reveal_reverse(list(range(1, 9))))
print("n = 7 :", reveal_reverse(list(range(1, 8))))

11 edge case: mọi cài đặt đúng
n = 8 : [1, 5, 2, 7, 3, 6, 4, 8]
n = 7 : [1, 6, 2, 5, 3, 7, 4]


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [22]:
def bug_no_rotate(deck):                      # quên đưa lá đáy lên đầu
    dq = deque()
    for x in sorted(deck, reverse=True):
        dq.appendleft(x)
    return list(dq)


def bug_rotate_wrong_direction(deck):         # xoay sai chiều: đầu xuống đáy thay vì đáy lên đầu
    dq = deque()
    for x in sorted(deck, reverse=True):
        if dq:
            dq.append(dq.popleft())
        dq.appendleft(x)
    return list(dq)


def bug_place_before_rotate(deck):            # đặt lá mới rồi mới xoay (sai thứ tự hai bước)
    dq = deque()
    for x in sorted(deck, reverse=True):
        dq.appendleft(x)
        dq.appendleft(dq.pop())
    return list(dq)


def bug_ascending_order(deck):                # duyệt tăng dần thay vì giảm dần
    dq = deque()
    for x in sorted(deck):
        if dq:
            dq.appendleft(dq.pop())
        dq.appendleft(x)
    return list(dq)


def bug_evens_then_odds(deck):                # chỉ nhớ "chẵn rồi lẻ", quên xoay khi n lẻ và các vòng sau
    s = sorted(deck)
    n = len(s)
    half = (n + 1) // 2
    res = [0] * n
    res[0::2] = s[:half]
    res[1::2] = s[half:]
    return res


def bug_forward_reveal_second(deck):          # mô phỏng xuôi nhưng xoay trước rồi mới lật
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        if len(dq) > 1:
            dq.append(dq.popleft())
        res[dq.popleft()] = x
    return res


def bug_return_sorted(deck):
    return sorted(deck)


def bug_skip_no_toggle_on_filled(deck):       # công tắc đảo cả khi gặp ô đã lấp (sai)
    n = len(deck)
    s = sorted(deck)
    res = [None] * n
    i = skip = placed = 0
    guard = 0
    while placed < n and guard < 10 * n * n + 10:
        guard += 1
        if res[i] is None and not skip:
            res[i] = s[placed]
            placed += 1
        skip = not skip
        i = (i + 1) % n
    return [v if v is not None else -1 for v in res]


BUGS = [bug_no_rotate, bug_rotate_wrong_direction, bug_place_before_rotate, bug_ascending_order, bug_evens_then_odds,
        bug_forward_reveal_second, bug_return_sorted, bug_skip_no_toggle_on_filled]
for bug in BUGS:
    bad = None
    for n in range(1, 10):
        d = list(range(1, n + 1))
        if bug(d) != reveal_reverse(d):
            bad = d
            break
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:30s} sai trên n = {len(bad)}: ra {bug(bad)}, đúng {reveal_reverse(bad)}, hợp lệ? {valid_answer(bad, bug(bad))}")

bug_no_rotate                  sai trên n = 3: ra [1, 2, 3], đúng [1, 3, 2], hợp lệ? False
bug_rotate_wrong_direction     sai trên n = 4: ra [1, 4, 3, 2], đúng [1, 3, 2, 4], hợp lệ? False
bug_place_before_rotate        sai trên n = 2: ra [2, 1], đúng [1, 2], hợp lệ? False
bug_ascending_order            sai trên n = 2: ra [2, 1], đúng [1, 2], hợp lệ? False
bug_evens_then_odds            sai trên n = 5: ra [1, 4, 2, 5, 3], đúng [1, 5, 2, 4, 3], hợp lệ? False
bug_forward_reveal_second      sai trên n = 2: ra [2, 1], đúng [1, 2], hợp lệ? False
bug_return_sorted              sai trên n = 3: ra [1, 2, 3], đúng [1, 3, 2], hợp lệ? False
bug_skip_no_toggle_on_filled   sai trên n = 2: ra [1, -1], đúng [1, 2], hợp lệ? False


### D. Vét cạn **mọi hoán vị** `n ≤ 8`

Với mỗi `n = 1..8`: duyệt mọi hoán vị của `1..n` (tổng cộng `46 233` hoán vị), đếm số hoán vị lật ra tăng dần: phải là **đúng 1**, và bằng đáp án của mọi cài đặt. Từ đó đáp án cho giá trị bất kỳ suy ra bằng ánh xạ hạng.

In [23]:
t0 = time.perf_counter()
unique_perm = {}
for n in range(1, 9):
    sols = [p for p in itertools.permutations(range(1, n + 1)) if reveals_increasing(p)]
    assert len(sols) == 1, (n, sols)                      # đáp án duy nhất
    unique_perm[n] = list(sols[0])
    base = list(range(1, n + 1))
    for name in IMPLS:
        assert IMPLS[name](base) == unique_perm[n], (name, n)
print("mọi hoán vị n ≤ 8: đúng 1 nghiệm cho mỗi n;", len(IMPLS), "cài đặt khớp;", round(time.perf_counter() - t0, 1), "s")
print({n: unique_perm[n] for n in (1, 2, 3, 4, 5)})
assert sum(math.factorial(n) for n in range(1, 9)) == 46233

mọi hoán vị n ≤ 8: đúng 1 nghiệm cho mỗi n; 9 cài đặt khớp; 0.1 s
{1: [1], 2: [1, 2], 3: [1, 3, 2], 4: [1, 3, 2, 4], 5: [1, 5, 2, 4, 3]}


In [24]:
# oracle cho giá trị bất kỳ (n <= 8): ánh xạ hạng qua nghiệm duy nhất; so với mọi cài đặt trên deck ngẫu nhiên có giá trị lớn
def oracle(deck):
    s = sorted(deck)
    return [s[r - 1] for r in unique_perm[len(deck)]]


rng = random.Random(95001)
for _ in range(1500):
    n = rng.randint(1, 8)
    d = rng.sample(range(1, 10 ** 6 + 1), n)
    want = oracle(d)
    assert reveal_brute(d) == want if n <= 6 else True
    check(d, want, brute=False)
print("1500 deck ngẫu nhiên n ≤ 8 (giá trị tới 10^6): oracle ánh xạ hạng khớp mọi cài đặt")

1500 deck ngẫu nhiên n ≤ 8 (giá trị tới 10^6): oracle ánh xạ hạng khớp mọi cài đặt


### E. Tính chất (n lớn, seed cố định)

(1) Kết quả là hoán vị của `deck` và lật ra tăng dần (`valid_answer`); (2) kết quả **không phụ thuộc thứ tự đầu vào**; (3) bốn cách khớp nhau; (4) lá đầu tiên (trên cùng) là `min(deck)`; (5) `reveal_positions_rec(n)` khớp mô phỏng cho mọi `n ≤ 1000`.

In [25]:
rng = random.Random(20240950)
t0 = time.perf_counter()
for _ in range(300):
    n = rng.randint(1, 1000)
    d = rng.sample(range(1, 10 ** 6 + 1), n)
    got = Solution().deckRevealedIncreasing(d)
    assert valid_answer(d, got)                                               # (1)
    sh = d[:]
    rng.shuffle(sh)
    assert Solution().deckRevealedIncreasing(sh) == got                       # (2)
    for name in ("forward", "halving", "skip"):
        assert IMPLS[name](d) == got, (name, n)                               # (3)
    assert got[0] == min(d)                                                   # (4)
for n in range(0, 1001):
    assert reveal_positions_rec(n) == reveal_positions_sim(n), n
print("300 deck ngẫu nhiên n ≤ 1000: 4 tính chất đúng; vị trí đệ quy = mô phỏng cho mọi n ≤ 1000;", round(time.perf_counter() - t0, 1), "s")

300 deck ngẫu nhiên n ≤ 1000: 4 tính chất đúng; vị trí đệ quy = mô phỏng cho mọi n ≤ 1000; 0.4 s


### F. Giới hạn

`n = 1000` với giá trị lớn nhất `10^6`, `deck` đã sắp tăng, giảm, và mọi `n` từ `1` đến `1000` với `deck = 1..n`.

In [26]:
t0 = time.perf_counter()
asc = list(range(1, 1001))
desc = asc[::-1]
big = list(range(10 ** 6 - 999, 10 ** 6 + 1))
for d in (asc, desc, big):
    got = Solution().deckRevealedIncreasing(d)
    assert valid_answer(d, got)
    for name in ("forward", "halving", "skip"):
        assert IMPLS[name](d) == got
for n in range(1, 1001):
    d = list(range(1, n + 1))
    got = reveal_reverse(d)
    assert reveals_increasing(got) and sorted(got) == d
    assert reveal_forward(d) == got and reveal_halving(d) == got
print("n = 1000 (3 dạng) và mọi n = 1..1000: hợp lệ và khớp;", round(time.perf_counter() - t0, 1), "s")

n = 1000 (3 dạng) và mọi n = 1..1000: hợp lệ và khớp; 0.2 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên đưa lá đáy lên đầu (mô phỏng ngược)

In [27]:
d = [17, 13, 11, 2, 3, 5, 7]
print("bug:", bug_no_rotate(d), "| lật ra:", reveal_order(bug_no_rotate(d)), "| đúng:", Solution().deckRevealedIncreasing(d))
assert not reveals_increasing(bug_no_rotate(d)) and reveals_increasing(Solution().deckRevealedIncreasing(d))

bug: [2, 3, 5, 7, 11, 13, 17] | lật ra: [2, 5, 11, 17, 7, 3, 13] | đúng: [2, 13, 3, 11, 5, 17, 7]


**Vì sao sai:** chỉ sắp xếp giảm dần rồi `appendleft` cho ra đúng thứ tự lá nhưng bỏ qua bước "lá kế xuống đáy". **Đúng:** trước mỗi lần đặt, đưa lá đáy lên đầu.

### Lỗi 2: xoay sai chiều

In [28]:
print("bug:", bug_rotate_wrong_direction(d), "| lật ra:", reveal_order(bug_rotate_wrong_direction(d)))
assert not reveals_increasing(bug_rotate_wrong_direction(d))

bug: [2, 17, 13, 11, 7, 5, 3] | lật ra: [2, 13, 7, 3, 11, 17, 5]


**Vì sao sai:** ngược quy trình "đầu → đáy" là "đáy → đầu" (`appendleft(pop())`), không phải `append(popleft())`.

### Lỗi 3: đặt lá trước, xoay sau

In [29]:
print("bug:", bug_place_before_rotate(d), "| lật ra:", reveal_order(bug_place_before_rotate(d)))
assert not reveals_increasing(bug_place_before_rotate(d))

bug: [7, 2, 13, 3, 11, 5, 17] | lật ra: [7, 13, 11, 17, 3, 2, 5]


**Vì sao sai:** lá mới phải nằm **trên cùng** sau khi xoay; nếu xoay sau thì lá mới bị đẩy xuống đáy. **Đúng:** xoay rồi mới đặt.

### Lỗi 4: duyệt giá trị tăng dần

In [30]:
print("bug:", bug_ascending_order(d), "| lật ra:", reveal_order(bug_ascending_order(d)))
assert not reveals_increasing(bug_ascending_order(d))

bug: [17, 3, 13, 5, 11, 2, 7] | lật ra: [17, 13, 11, 7, 5, 3, 2]


**Vì sao sai:** mô phỏng ngược phải bắt đầu từ lá được lật **cuối cùng** (lớn nhất). **Đúng:** `sorted(deck, reverse=True)`.

### Lỗi 5: chỉ nhớ "chẵn rồi lẻ"

In [31]:
d7 = list(range(1, 8))
print("bug:", bug_evens_then_odds(d7), "| đúng:", reveal_reverse(d7))
assert bug_evens_then_odds(d7) != reveal_reverse(d7) and not reveals_increasing(bug_evens_then_odds(d7))

bug: [1, 5, 2, 6, 3, 7, 4] | đúng: [1, 6, 2, 5, 3, 7, 4]


**Vì sao sai:** sau vòng đầu các lá lẻ còn lại phải được xử lý **đệ quy** (và xoay khi `n` lẻ), không phải giữ nguyên thứ tự.

### Lỗi 6 (bonus): mô phỏng xuôi nhưng xoay trước khi lật

In [32]:
print("bug:", bug_forward_reveal_second(d), "| đúng:", reveal_forward(d))
assert bug_forward_reveal_second(d) != reveal_forward(d) and not reveals_increasing(bug_forward_reveal_second(d))

bug: [7, 2, 13, 3, 11, 5, 17] | đúng: [2, 13, 3, 11, 5, 17, 7]


**Vì sao sai:** quy trình đề: **lật trước**, rồi mới chuyển lá kế xuống đáy.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "sắp xếp để một quy trình cố định cho ra thứ tự mong muốn" → quy trình chỉ phụ thuộc **vị trí**, không phụ thuộc giá trị ⇒ mô phỏng trên chỉ số, hoặc **đảo ngược** quy trình.

**Mẹo:**

1. Mô phỏng trên **chỉ số** rồi gán giá trị đã sắp xếp — dùng lại cho mọi bài "xếp theo luật".
2. Đảo ngược quy trình: đảo thứ tự các bước **và** đảo từng bước.
3. `deque` cho hai đầu `O(1)`; `list.pop(0)` / `insert(0, x)` là `O(n)` (vẫn qua với `n ≤ 1000`).
4. Viết **hàm mô phỏng nguyên văn** làm oracle/kiểm tra hợp lệ.
5. Với đáp án duy nhất, vét cạn mọi hoán vị nhỏ để chứng minh tính duy nhất và so sánh.

**Họ bài cùng pattern:** Josephus (Find the Winner of the Circular Game, 1823), Design Circular Queue (622), Rotate List (61), Rotate Array (189).

## 15. Biến thể

1. **Lật theo thứ tự giảm dần:** gán các giá trị sắp xếp giảm dần.
2. **Chuyển `k` lá xuống đáy** sau mỗi lần lật (tổng quát `k = 1`).
3. **Giá trị trùng:** đáp án không còn duy nhất; dùng hàm kiểm tra không giảm (`<=`).
4. **Tính trực tiếp vị trí lá thứ `k`** (không dựng cả bộ bài).

Cài biến thể 1, 2 và 3, kiểm bằng vét cạn và hàm hợp lệ:

In [33]:
def reveal_order_k(order, k):
    """Lật lá trên cùng; rồi chuyển k lá trên cùng kế tiếp (lần lượt) xuống đáy."""
    dq = deque(order)
    out = []
    while dq:
        out.append(dq.popleft())
        for _ in range(min(k, len(dq))):
            dq.append(dq.popleft())
    return out


def deck_k(deck, k):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        res[dq.popleft()] = x
        for _ in range(min(k, len(dq))):
            dq.append(dq.popleft())
    return res


def deck_decreasing(deck):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck, reverse=True):
        res[dq.popleft()] = x
        if dq:
            dq.append(dq.popleft())
    return res


for k in (0, 1, 2, 3):
    for n in range(1, 8):
        base = list(range(1, n + 1))
        got = deck_k(base, k)
        out = reveal_order_k(got, k)
        assert out == base, (k, n)
        sols = [p for p in itertools.permutations(base) if reveal_order_k(p, k) == base]
        assert sols == [tuple(got)], (k, n, sols)                          # duy nhất và khớp vét cạn
assert deck_k([5, 3, 9, 1], 1) == reveal_reverse([5, 3, 9, 1])
for n in range(1, 8):
    base = list(range(1, n + 1))
    assert reveal_order(deck_decreasing(base)) == base[::-1]
# giá trị trùng: sắp không giảm vẫn hợp lệ theo nghĩa "không giảm"
dup = [3, 1, 3, 2, 2, 1]
got = reveal_reverse(dup)
out = reveal_order(got)
assert sorted(got) == sorted(dup) and all(out[i] <= out[i + 1] for i in range(len(out) - 1))
print("k = 0..3 duy nhất và khớp vét cạn; giảm dần đúng; giá trị trùng cho dãy lật không giảm:", got, "->", out)

k = 0..3 duy nhất và khớp vét cạn; giảm dần đúng; giá trị trùng cho dãy lật không giảm: [1, 2, 1, 3, 2, 3] -> [1, 1, 2, 2, 3, 3]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi hoán vị | oracle brute | `O(n!·n)` | `O(n)` | kiểm chứng `n ≤ 8` |
| 2. Mô phỏng ngược (deque) | đảo quy trình | `O(n log n)` | `O(n)` | **đáp án chính** |
| 3. Mô phỏng xuôi chỉ số | tính `π` rồi gán | `O(n log n)` | `O(n)` | dễ chứng minh |
| 4. Đệ quy giảm một nửa | chẵn rồi bài toán con | `O(n log n)` | `O(n)` | công thức vị trí |
| 5. Mảng vòng + công tắc | bỏ qua ô trống | `O(n log n)` | `O(n)` | không dùng deque |

Mọi cách khớp nghiệm duy nhất của vét cạn (`n ≤ 8`) và khớp nhau tới `n = 1000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [34]:
print(Solution().deckRevealedIncreasing([4, 3, 2, 1]), Solution().deckRevealedIncreasing([9, 7, 5]), reveal_order([1, 3, 2, 4]), reveal_positions_sim(6))

[1, 3, 2, 4] [5, 9, 7] [1, 2, 3, 4] [0, 2, 4, 1, 5, 3]


### Bài 2 (tính tay)

`deck = [10, 20, 30, 40, 50, 60]`. (a) Viết thứ tự lật các vị trí `π(6)`. (b) Viết bộ bài trả về. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện mô phỏng xuôi trên chỉ số:

In [35]:
def forward_todo(deck):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        pass            # TODO 1: gán x vào vị trí lấy ra từ đầu hàng đợi
        pass            # TODO 2: nếu hàng đợi còn phần tử thì chuyển phần tử đầu xuống đáy
    return res


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(forward_todo(d) == w for d, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mô phỏng ngược với deque**:

In [36]:
class MySolution:
    def deckRevealedIncreasing(self, deck):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().deckRevealedIncreasing(list(d)) == w for d, w in examples)

### Bài 5 (mini challenge): chuyển `k` lá xuống đáy

Sau mỗi lần lật, chuyển lần lượt `k` lá trên cùng kế tiếp xuống đáy (nếu đủ lá). Viết `deck_revealed_k(deck, k)` trả bộ bài lật ra tăng dần, và kiểm bằng `reveal_order_k` ở mục 15.

In [37]:
def deck_revealed_k(deck, k):
    pass            # TODO


# Khi làm xong:
# for k in (1, 2, 3):
#     d = [9, 4, 7, 1, 8, 2, 6]
#     assert reveal_order_k(deck_revealed_k(d, k), k) == sorted(d)

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: nhớ `n = 4` cho `[1,3,2,4]`.
- Mức 2: `[9,7,5]` tương ứng `n = 3` ⇒ vị trí lật `0,2,1`.
- Mức 3: kết quả `[1, 3, 2, 4] [5, 9, 7] [1, 2, 3, 4] [0, 2, 4, 1, 5, 3]`; với `n = 6` các vị trí chẵn `0,2,4` lật trước, rồi `1`, `5`, `3`.

**Bài 2**

- Mức 1: mô phỏng hàng đợi chỉ số `[0..5]`.
- Mức 2: lật 0, xuống đáy 1; lật 2, xuống đáy 3; lật 4, xuống đáy 5.
- Mức 3: sau vòng đầu còn `[1,3,5]`; xem mục 19 để so khớp.

**Bài 3**

- Mức 1: `res[dq.popleft()] = x`.
- Mức 2: `if dq: dq.append(dq.popleft())`.
- Mức 3: giống Cách 3 ở mục 7.

**Bài 4**

- Mức 1: sắp xếp giảm dần.
- Mức 2: mỗi lá: nếu `dq` không rỗng thì `appendleft(pop())`; rồi `appendleft(x)`.
- Mức 3: trả `list(dq)`.

**Bài 5**

- Mức 1: tổng quát hoá Cách 3: thay `if dq:` bằng vòng lặp `k` lần.
- Mức 2: mỗi vòng `for _ in range(min(k, len(dq)))`.
- Mức 3: `res[dq.popleft()] = x` rồi xoay `k` lần.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [38]:
vals = (Solution().deckRevealedIncreasing([4, 3, 2, 1]), Solution().deckRevealedIncreasing([9, 7, 5]), reveal_order([1, 3, 2, 4]), reveal_positions_sim(6))
print(vals)
assert vals == ([1, 3, 2, 4], [5, 9, 7], [1, 2, 3, 4], [0, 2, 4, 1, 5, 3])
print("Bài 1 OK")

([1, 3, 2, 4], [5, 9, 7], [1, 2, 3, 4], [0, 2, 4, 1, 5, 3])
Bài 1 OK


### Bài 2

Hàng đợi `[0,1,2,3,4,5]`: lật 0 → `[2,3,4,5,1]`; lật 2 → `[4,5,1,3]`; lật 4 → `[1,3,5]`; lật 1 → `[5,3]`; lật 5 → `[3]`; lật 3. Nên `π(6) = [0,2,4,1,5,3]`.

In [39]:
pi6 = reveal_positions_sim(6)
print("π(6) =", pi6)
assert pi6 == [0, 2, 4, 1, 5, 3]
ans2 = Solution().deckRevealedIncreasing([10, 20, 30, 40, 50, 60])
print("bộ bài:", ans2)
assert ans2 == [10, 40, 20, 60, 30, 50] and reveal_order(ans2) == [10, 20, 30, 40, 50, 60]
print("Bài 2 OK")

π(6) = [0, 2, 4, 1, 5, 3]
bộ bài: [10, 40, 20, 60, 30, 50]
Bài 2 OK


### Bài 3

In [40]:
def forward_answer(deck):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        res[dq.popleft()] = x
        if dq:
            dq.append(dq.popleft())
    return res


assert all(forward_answer(d) == w for d, w in examples)
for n in range(1, 60):
    assert forward_answer(list(range(n, 0, -1))) == reveal_reverse(list(range(1, n + 1)))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [41]:
class MySolutionAnswer:
    def deckRevealedIncreasing(self, deck):
        dq = deque()
        for x in sorted(deck, reverse=True):
            if dq:
                dq.appendleft(dq.pop())
            dq.appendleft(x)
        return list(dq)


assert all(MySolutionAnswer().deckRevealedIncreasing(list(d)) == w for d, w in examples)
rng = random.Random(4)
for _ in range(100):
    d = rng.sample(range(1, 10 ** 6), rng.randint(1, 200))
    assert MySolutionAnswer().deckRevealedIncreasing(d) == reveal_forward(d)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [42]:
def deck_revealed_k_answer(deck, k):
    n = len(deck)
    res = [0] * n
    dq = deque(range(n))
    for x in sorted(deck):
        res[dq.popleft()] = x
        for _ in range(min(k, len(dq))):
            dq.append(dq.popleft())
    return res


for k in (1, 2, 3):
    d = [9, 4, 7, 1, 8, 2, 6]
    assert reveal_order_k(deck_revealed_k_answer(d, k), k) == sorted(d)
for k in (0, 1, 2, 3):
    for n in range(1, 7):
        base = list(range(1, n + 1))
        assert deck_revealed_k_answer(base, k) == deck_k(base, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (xếp để quy trình cố định cho ra thứ tự mong muốn → đảo ngược quy trình):**

```python
class Solution:
    def deckRevealedIncreasing(self, deck: List[int]) -> List[int]:
        dq = deque()
        for x in sorted(deck, reverse=True):
            if dq:
                dq.appendleft(dq.pop())     # đảo "lá kế xuống đáy"
            dq.appendleft(x)                # đảo "lật lá trên cùng"
        return list(dq)
```

**Checklist trước khi submit:**

- [ ] Sắp xếp **giảm dần** khi mô phỏng ngược?
- [ ] Xoay (`appendleft(pop())`) **trước** khi đặt lá mới, và chỉ khi `dq` không rỗng?
- [ ] Trả `list(dq)` (không phải `dq`)?
- [ ] Hiểu vì sao đáp án duy nhất (giá trị khác nhau, thứ tự lật bắt buộc tăng)?
- [ ] Đã thử `[17,13,11,2,3,5,7] → [2,13,3,11,5,17,7]`, `[1,1000]`, `n = 1`, `n` lẻ (`3`, `5`, `7`)?
- [ ] Đúng tên `class Solution`, `deckRevealedIncreasing(self, deck)`, không in debug.